# Figure 3 — CIFAR-10 Test Error
Reproduces Figure 3 (left) of Huang et al. 2016.

1. Run **Mount & Copy** cell first.
2. Run **Smoke Test** to confirm setup.
3. Run one training cell per tab — Tab 1: stochastic, Tab 2: baseline.

> Runtime → Change runtime type → **A100 GPU**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
ROOT = '/content/drive/MyDrive/stoch_depth'

for fname in ['model.py', 'data.py', 'train.py', 'evaluate.py']:
    shutil.copy(f'{ROOT}/code/{fname}', f'/content/{fname}')
    print(f'Copied {fname}')


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Smoke Test
Run first (~2 min). You should see epoch progress printed.

In [ ]:
import os
ROOT = '/content/drive/MyDrive/stoch_depth'
OUT = f'{ROOT}/smoke_test'

!python /content/train.py \
    --dataset cifar10 --p_L 0.5 --survival_mode linear \
    --epochs 5 --max_batches 20 \
    --data_root /content/data \
    --num_workers 2 \
    --out_dir $OUT


## Train — Stochastic Depth (Tab 1)

In [ ]:
import os
ROOT = '/content/drive/MyDrive/stoch_depth'
CKPT = f'{ROOT}/fig3/stochastic/checkpoints/cifar10_pL0.5_modelinear_n18_last.pt'
RESUME = f'--resume {CKPT}' if os.path.exists(CKPT) else ''
OUT = f'{ROOT}/fig3/stochastic'

!python /content/train.py \
    --dataset cifar10 --p_L 0.5 --survival_mode linear \
    --epochs 500 --test_every_epoch \
    --data_root /content/data \
    --num_workers 2 \
    --out_dir $OUT \
    $RESUME


## Train — Constant Depth Baseline (Tab 2)

In [ ]:
import os
ROOT = '/content/drive/MyDrive/stoch_depth'
CKPT = f'{ROOT}/fig3/baseline/checkpoints/cifar10_pL1.0_modeconstant_n18_last.pt'
RESUME = f'--resume {CKPT}' if os.path.exists(CKPT) else ''
OUT = f'{ROOT}/fig3/baseline'

!python /content/train.py \
    --dataset cifar10 --p_L 1.0 --survival_mode constant \
    --epochs 500 --test_every_epoch \
    --data_root /content/data \
    --num_workers 2 \
    --out_dir $OUT \
    $RESUME


## Evaluate (run after both training runs finish)

In [ ]:
import os
ROOT = '/content/drive/MyDrive/stoch_depth'
CKPT_S = f'{ROOT}/fig3/stochastic/checkpoints/cifar10_pL0.5_modelinear_n18_best.pt'
CKPT_B = f'{ROOT}/fig3/baseline/checkpoints/cifar10_pL1.0_modeconstant_n18_best.pt'

print('--- Stochastic ---')
!python /content/evaluate.py --checkpoint $CKPT_S --dataset cifar10 --p_L 0.5 --data_root /content/data

print('--- Baseline ---')
!python /content/evaluate.py --checkpoint $CKPT_B --dataset cifar10 --p_L 1.0 --data_root /content/data
